#Part 3:

In [0]:
import pandas as pd
import numpy as np
import math

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

##Dataframes:

###In Part 2. Ques 3. c. result csv was uploaded manually:

In [0]:
Q3C_CSV = "/Volumes/workspace/bde/assignment2/Part_2_3_c.csv"
q3c_df = pd.read_csv(Q3C_CSV)
print("Q3 (reference) shape:", q3c_df.shape)
display(q3c_df.info())

Q3 (reference) shape: (10000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   color             10000 non-null  object 
 1   pu_borough        10000 non-null  object 
 2   do_borough        10000 non-null  object 
 3   month_num         10000 non-null  int64  
 4   day_of_week       10000 non-null  object 
 5   hour_of_day       10000 non-null  int64  
 6   avg_total_amount  10000 non-null  float64
dtypes: float64(1), int64(2), object(4)
memory usage: 547.0+ KB


###In Part 1. Ques 7 trips_final table was saved in Databricks:

In [0]:
trip_df = spark.table("bde.trips_final").sample(fraction=0.01, seed=42).toPandas()
trip_df.count()

color                    9596491
vendor_id                9596491
ratecode_id              9596491
payment_type             9596491
store_and_fwd_flag       9098977
pickup_datetime          9596491
dropoff_datetime         9596491
pu_location_id           9596491
do_location_id           9596491
passenger_count          9596491
trip_distance            9596491
fare_amount              9596491
extra                    9596491
mta_tax                  9596491
improvement_surcharge    8929634
tip_amount               9596491
tolls_amount             9596491
congestion_surcharge     2443606
airport_fee               637264
ehail_fee                     34
total_amount             9596491
trip_type                 774977
pu_borough               9596491
pu_zone                  9596491
pu_service_zone          9596491
do_borough               9596491
do_zone                  9596491
do_service_zone          9596491
dtype: int64

In [0]:
trip_df["pickup_datetime"] = pd.to_datetime(trip_df["pickup_datetime"], errors="coerce")
trip_df["month_num"] = trip_df["pickup_datetime"].dt.month
trip_df["day_of_week"] = trip_df["pickup_datetime"].dt.day_name()
trip_df["hour_of_day"] = trip_df["pickup_datetime"].dt.hour

In [0]:
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9596491 entries, 0 to 9596490
Data columns (total 31 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   color                  object        
 1   vendor_id              int32         
 2   ratecode_id            int32         
 3   payment_type           int32         
 4   store_and_fwd_flag     object        
 5   pickup_datetime        datetime64[ns]
 6   dropoff_datetime       datetime64[ns]
 7   pu_location_id         int32         
 8   do_location_id         int32         
 9   passenger_count        int32         
 10  trip_distance          float64       
 11  fare_amount            float64       
 12  extra                  float64       
 13  mta_tax                float64       
 14  improvement_surcharge  float64       
 15  tip_amount             float64       
 16  tolls_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee           

###Train & Test Split:

In [0]:
train_df = trip_df[trip_df["pickup_datetime"] < "2024-10-01"].copy()

test_df  = trip_df[(trip_df["pickup_datetime"] >= "2024-10-01") & (trip_df["pickup_datetime"] < "2025-01-01")].copy()

print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)

Train shape: (9497355, 31) | Test shape: (99135, 31)


Note: Train df has data till the end of September 2024. Test df has data from start of October 2024 to end of December 2024.

##Features:

In [0]:
cat_cols = ['color','pu_borough','do_borough','day_of_week']
num_cols = ['passenger_count','trip_distance','month_num','hour_of_day','tip_amount','payment_type']
final_cols = cat_cols + num_cols

target_col = 'total_amount'

##Model Preparation:

In [0]:
X_train = train_df[final_cols]
y_train = train_df[target_col]

X_test = test_df[final_cols]
y_test = test_df[target_col]

In [0]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9497355 entries, 0 to 9596490
Data columns (total 10 columns):
 #   Column           Dtype  
---  ------           -----  
 0   color            object 
 1   pu_borough       object 
 2   do_borough       object 
 3   day_of_week      object 
 4   passenger_count  int32  
 5   trip_distance    float64
 6   month_num        int64  
 7   hour_of_day      int64  
 8   tip_amount       float64
 9   payment_type     int32  
dtypes: float64(2), int32(2), int64(2), object(4)
memory usage: 724.6+ MB


##Preprocessing:

In [0]:
preproc = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat_cols),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop'
)

##Baseline:

In [0]:
baseline_test_df = test_df[[ 'color','pu_borough','do_borough','day_of_week','month_num','hour_of_day','total_amount']].copy()

merged_df = pd.merge(
    baseline_test_df,
    q3c_df,
    on=[ 'color','pu_borough','do_borough','day_of_week','month_num','hour_of_day'],
    how="left"
)

In [0]:
y_pred_baseline = merged_df["avg_total_amount"].fillna(merged_df["avg_total_amount"].mean())

baseline_rmse = mean_squared_error(merged_df["total_amount"], y_pred_baseline, squared=False)

baseline_rmse

/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


26.60034960039372

##Ridge Regression Model:

In [0]:
ridge = Ridge(alpha=1.0, random_state=42)

pipe = Pipeline(steps=[
    ('pre', preproc),
    ('model', ridge)
])

pipe.fit(X_train, y_train)

y_train_pred = pipe.predict(X_train)
y_test_pred = pipe.predict(X_test)

train_ridge_rmse = math.sqrt(mean_squared_error(y_train, y_train_pred))
print("Ridge Train RMSE :", round(train_ridge_rmse, 6))

test_ridge_rmse = math.sqrt(mean_squared_error(y_test, y_test_pred))
print("Ridge Test RMSE :", round(test_ridge_rmse, 6))

Ridge Train RMSE : 190.893137
Ridge Test RMSE : 12.992214


##Random Forest Regressor:

In [0]:
rf = RandomForestRegressor(
    n_estimators=75,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf_pipe = Pipeline(steps=[
    ('pre', preproc),
    ('model', rf)
])

rf_pipe.fit(X_train, y_train)

y_train_pred = rf_pipe.predict(X_train)
y_test_pred = rf_pipe.predict(X_test)


train_rf_rmse = math.sqrt(mean_squared_error(y_train, y_train_pred))
print("Random Forest Train RMSE :", round(train_rf_rmse, 6))

test_rf_rmse = math.sqrt(mean_squared_error(y_test, y_test_pred))
print("Random Forest Test RMSE :", round(test_rf_rmse, 6))

Random Forest Train RMSE : 98.021944
Random Forest Test RMSE : 11.013514


In [0]:
print("Baseline Test RMSE :", round(baseline_rmse, 6))
print("Ridge Test RMSE :", round(test_ridge_rmse, 6))
print("Random Forest Test RMSE :", round(test_rf_rmse, 6))

Baseline Test RMSE : 26.60035
Ridge Test RMSE : 12.992214
Random Forest Test RMSE : 11.013514


Analysis: Random Forest is selected as the best model because, despite its higher complexity and 12-minute runtime, it significantly outperforms Ridge Regression in predictive accuracy (Test RMSE 11.01 vs. 12.99). While Ridge is faster and simpler, the accuracy gain from Random Forest justifies the extra cost. Moreover, Random Forest dramatically beats the baseline RMSE of 26.60, confirming that it captures real, nontrivial structure in the data. 
For the full dataset, Random Forest is the better model because it generalizes well and delivers much lower RMSE. However, if compute costs and processing time are constraints, Ridge Regression remains an acceptable fallback. A hybrid approach could also work, train Ridge for quick, cheap baselines, and use Random Forest in batch mode where high accuracy is needed.